## From Building Blocks to U-Net - Understanding Architectural Innovation

### Part 1: Understanding U-Net Architecture (15 points)

U-Net was introduced by Ronneberger et al. (2015) for biomedical image segmentation. The architecture won the ISBI cell tracking challenge 2015 by a large margin.

![U-Net Architecture](https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/u-net-architecture.png)
*Figure 1: Original U-Net architecture (Ronneberger et al., 2015)*

### Starter Code: Basic U-Net Implementation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

**Q1.1 (5 points):** Identify and explain the three major architectural components/concepts from our previous lectures that U-Net combines. For each component:
- Name the concept
- Explain how it's implemented in U-Net
- Describe what problem it solves

**Name of Concept 1**: Encoding and Decoding Information with Bottleneck

**Description of how Concept 1 is implemented in U-Net**: The first part of the U is similar to the encoder part of the EncoderGRU because it takes an image and applies a series of convolutions and max poolings to it to compress it from 572^2 image to 32^2 images. The 32^2 images act as the information bottleneck that forces the model to learn the important patterns of the image while discarding the noise. Then the model uses the up convolution to expand the images back to a recognizable image so that the patterns can be applied to the locations in the image, similar to a decoder outputting the semantic meaning of a text to a different language.

**Problem Concept 1 solves**: The problem this solves is being able to combine the classification aspect of CNN with the spatial locality of the image. In a traditional CNN the down convolution is used to turn an image into a classification score. However, the innovation of the U-Net is to Up convolution these classification scores and translate them back to the original image so it can recognize where the patterns are in the image.

-----

**Name of Concept 2**: Skip Connections / concatenation skip

**Description of how Concept 2 is implemented in U-Net**: The skip connections used in the U-Net (copy and crop connections) are similar to how ResNet uses residual connections. Both pass both the modified/encoded information and the earlier, less-processed features to the next layer so that layer can use this combined information to learn more accurately. These techniques also similarly allow for the networks to grow in depth without losing the original gradient / spatial detail.

**Problem Concept 2 solves**: This solves the problem of knowing where the high level encoded features exist within the low level, high-resolution encoder features. It allows for the decoder to have information on the edges and fine spatial detail captured by the encoder so it can accurately place the high level features on it. This prevents the encoded/decoded image from being completely different than the original image.


-----

**Name of Concept 3**: Channel Concatenation 

**Description of how Concept 3 is implemented in U-Net**: The concept of channel concatenation is used in both U-Net and Inception Net to combine information from different layer of the network. Inception net uses it to mix different convolutions sizes to get more nonlinearity from different convolutional filter while U-Net combines the channels to get information from encoded/decoded features and low level unprocessed features.

**Problem Concept 3 solves**:  This solves the problem of having to choose between the low level, high-resolution features and highly information dense patterns learned by the network. Instead of dropping or not using one, they are combined through concatenation to get the most information from both.

**Q1.2 (5 points):** Skip connections in U-Net use **concatenation** rather than **addition** (as in ResNet).
- What are the implications of this design choice?
- How does this affect the number of parameters?
- When might addition be preferred over concatenation?

**Implications of having skip connections in U-Net use concatenation**: The implication of concatenation is that instead of having to formulaicly add the two path of information ( encoded/decoded features / skip features) into one feature path like is done with addition in ResNet, the two sets of features are both used by the Double convolution blocks so that the model can learn from both.

**Effect of concatenation on number of parameters**: The concatenation effectively doubles the number of channels that the next block has so it will also double the number of parameters that the convolutional needs. This has the effect of increasing total features just in the convolution block, which is overall not a ton of features but a modest increase.

**When addition might be preferred over concatenation**: Addition is preferred when the features form the two paths are similar and there is no need to look at both. This will save memory and computational power.

**Q1.3 (5 points):** Critical Analysis:
- What are the potential limitations of the original U-Net architecture?
- How might U-Net struggle with very high-resolution images?
- Propose at least two modifications that could address these limitations.

**Potential limitations of the original U-Net architecture**: The original U-Net is convolutional kernel size being set to 3 which means the model will learn localized patterns but may struggle to find patterns over the entire image. Also, the U-Net has a lot of parameters which can cause the model to overfit and be computationally expensive.

**How U-Net might struggle with very high-resolution images**: High resolution images have more pixels per area which will cause U-Net to struggle more learning image wide patterns because the 3x3 convolutions will cover less of the image area than in a low resolution image. Also, the model will have a lot more parameters to deal with compressing the increased resolution image to bottleneck size which can cause computation costs. 

**Proposed modifications**: Add Inception net style parallel paths of different convolutional sizes that can use both local and more image spanning patterns. Additionally, adding a 1X1 convolutional layer to reduce channels like in NiN can improve the parameter efficiency without sacrificing accuracy.

### Part 2: Architectural Variants (15 points)

#### Variant 1: U-Net++ (Nested U-Net)

U-Net++ introduces nested skip connections to reduce the semantic gap between encoder and decoder features.

In [ ]:
class UNetPlusPlus(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # Define the nested structure
        # X^{0,0} -> X^{1,0} -> X^{2,0} -> X^{3,0} -> X^{4,0}
        #    |         |         |         |
        # X^{0,1} -> X^{1,1} -> X^{2,1} -> X^{3,1}
        #    |         |         |
        # X^{0,2} -> X^{1,2} -> X^{2,2}
        #    |         |
        # X^{0,3} -> X^{1,3}
        #    |
        # X^{0,4}

        nb_filter = [32, 64, 128, 256, 512]

        # Nested skip connections would be implemented here.
        # This is a simplified structure for illustration.
        pass

#### Variant 2: V-Net (Volumetric Convolutional Neural Networks)

V-Net extends U-Net to 3D volumetric data and adds residual connections within each stage.

In [ ]:
class VNetBlock(nn.Module):

    def __init__(self, in_channels, out_channels, num_convs):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv3d(in_channels if i == 0 else out_channels,
                     out_channels, kernel_size=3, padding=1)
            for i in range(num_convs)
        ])
        self.activation = nn.PReLU(out_channels)

        self.residual = nn.Conv3d(in_channels, out_channels, kernel_size=1) \
                        if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)

        out = x
        for conv in self.convs:
            out = self.activation(conv(out))

        return out + residual

**Q3.1 (5 points):** U-Net++ Analysis:
- What problem does the nested skip connection architecture solve?
- How does it reduce the "semantic gap"?
- What is the computational cost compared to standard U-Net?

**Problem nested skip connection architecture solves**: The nested skip connection architecture solves the issues of concatenating the shallow encoder features with the more information-dense decoder features, since these two sets of features are differ in their representation. The nested skip connections are used to apply convolutional blocks to the encoder features to transform them into a representation that is easier to combine with the encoded/decoded image, thus improving the accuracy of the up convolution to a larger image.

**How nested skip connection architecture reduces semantic gap**: The nested skip connection architecture improves the semantic gap by applying a convolutional block to turn the encoder features into a dense representation similar to the features that have already been through the bottleneck. However, since these features have not been through the bottleneck and just goes through convolution, it is able to retain the spatial features that can be combined with the encoded/decoded image to create a spatially relevant image with the high level features like in the original U-net. 

**Computational cost of U-Net with nested skip connection architecture**: The parameter count does increase, but not by  a large amount. The normal U-net concatenation causes the parameters of the convolutions in decoder blocks to double, while the convolutional parameters from unet++ nested skip connections adds a lower amount of parameters than double, which is a good tradeoff from the increases in accuracy from lower the semantic gap. 

**Q3.2 (5 points):** V-Net Contributions:
- Besides 3D extension, what are the key innovations in V-Net?
- Why are residual connections particularly important for volumetric data?
- How does the Dice loss implementation differ for 3D data?

**Key innovations in V-Net**: The key innovations in V-net are that it adds Resnet like residual connection at each layer along the encoding and decoding paths (each encoder/ decoder block) down the V instead of just across the V which allows for faster convergence in training. Furthermore, it uses Dice Loss instead of standard cross entropy loss which helps solve the class imbalance problem seen in medical industry. 

**Why residual connections are important for volumetric data**: Residual connections help combat the vanishing gradient problem. Since volumetric networks need much higher level features to represent the data, the network must be significantly deeper than a 2D processing network. As a result we see the vanishing gradient problem when going through a large number of 3D convolutional layers. The residual connections help maintain the gradient of the loss function and help prevent the network from overfitting.

**How Dice loss differs for 3D data**: In V-net Dice loss flattens the 3D tensor of data into a 1D vector and applies a soft-differentiable volumetric formulation to help with gradient stability. It also squares the denominator terms, which helps push the network to more defined boundaries faster in training.

**Q3.3 (5 points):** Comparative Analysis:
Create a comparison table with at least 5 criteria comparing:
- Standard U-Net
- U-Net++
- V-Net

Consider: parameter count, memory usage, suitable applications, training difficulty, and inference speed.

| Criterion | Standard U-Net | U-Net++ | V-Net |
| ----- | ----- | ----- | ----- |
| Parameter Count | Moderate parameters count (~7.8 M) |  Higher moderate parameters count (~9 M) | High parameters count (~50 M) |
| Memory Usage | Moderate | High: due to nested skip connection storing more image | Very high: grows cubically due to 3D data |
| Suitable Applications | General 2D segmentation of cells and xray images | Higher accuracy 2D segmentation with fine boundaries and objects of varying size | 3D volumetric segmentation (MRI / CT scans) |
| Training Difficulty | Easy: Stable and works on small datasets  | Moderate: Needs more compute to due "deep-supervision" needing more tuning | Hardest: 3D data is scarce and contains extreme class imbalance (residual help) |
| Inference Speed | Fastest | Slower than U-Net, but can be increased with pruning | Slowest as it must process entire 3D volume of data (but better than slice by slice) |

## Resources

### Papers
- [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)
- [UNet++: A Nested U-Net Architecture](https://arxiv.org/abs/1807.10165)
- [V-Net: Fully Convolutional Neural Networks for Volumetric Medical Image Segmentation](https://arxiv.org/abs/1606.04797)

### Implementations
- [PyTorch U-Net](https://github.com/milesial/Pytorch-UNet)
- [Segmentation Models PyTorch](https://github.com/qubvel/segmentation_models.pytorch)
- [MONAI (Medical Imaging)](https://monai.io/)